In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import joblib
import os
import json

c:\Users\84352\Desktop\diabetes-prediction-webapp\notebooks\notebooks


In [ ]:
DATA_PATH = "../data/raw/diabetes.csv"
df = pd.read_csv(DATA_PATH)

print(df.shape)
print(df['Outcome'].value_counts(normalize=True))  # Check imbalance
df.head()

(768, 9)
Outcome
0    0.651042
1    0.348958
Name: proportion, dtype: float64


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [11]:
# Các cột có giá trị 0 không hợp lý
cols_to_impute = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

# Thay 0 → NaN
for col in cols_to_impute:
    df[col] = df[col].replace(0, np.nan)

# Tách X, y
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split (stratify để giữ tỷ lệ class)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (614, 8)
Test shape: (154, 8)


In [12]:
# Imputer median (tốt cho dữ liệu skewed)
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Scale (dùng cho LogReg, nhưng RF cũng ok)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

In [13]:
models = {
    'LogisticRegression': LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000),
    'DecisionTree': DecisionTreeClassifier(class_weight='balanced', random_state=42),
    'RandomForest': RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=100)
}

for name, model in models.items():
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_prob))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


=== LogisticRegression ===
              precision    recall  f1-score   support

           0       0.82      0.75      0.79       100
           1       0.60      0.70      0.65        54

    accuracy                           0.73       154
   macro avg       0.71      0.73      0.72       154
weighted avg       0.75      0.73      0.74       154

ROC-AUC: 0.8125925925925926
Confusion Matrix:
 [[75 25]
 [16 38]]

=== DecisionTree ===
              precision    recall  f1-score   support

           0       0.75      0.84      0.79       100
           1       0.62      0.48      0.54        54

    accuracy                           0.71       154
   macro avg       0.68      0.66      0.67       154
weighted avg       0.70      0.71      0.70       154

ROC-AUC: 0.6607407407407406
Confusion Matrix:
 [[84 16]
 [28 26]]

=== RandomForest ===
              precision    recall  f1-score   support

           0       0.79      0.84      0.81       100
           1       0.66      0.57

In [14]:
# Tune RandomForest (GridSearch đơn giản)
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 15],
    'min_samples_split': [2, 5],
    'class_weight': ['balanced']
}

rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

print("Best params:", grid_search.best_params_)
print("Best ROC-AUC (CV):", grid_search.best_score_)

# Model cuối
best_model = grid_search.best_estimator_

# Đánh giá cuối trên test
y_pred_final = best_model.predict(X_test_scaled)
y_prob_final = best_model.predict_proba(X_test_scaled)[:, 1]
print("\nFinal RandomForest:")
print(classification_report(y_test, y_pred_final))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_final))

Best params: {'class_weight': 'balanced', 'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 100}
Best ROC-AUC (CV): 0.8313178294573644

Final RandomForest:
              precision    recall  f1-score   support

           0       0.83      0.81      0.82       100
           1       0.66      0.69      0.67        54

    accuracy                           0.77       154
   macro avg       0.74      0.75      0.75       154
weighted avg       0.77      0.77      0.77       154

ROC-AUC: 0.8290740740740741


In [ ]:
os.makedirs("../backend/assets", exist_ok=True)

# Save model, scaler, imputer
joblib.dump(best_model, "../backend/assets/diabetes_model.pkl")
joblib.dump(scaler, "../backend/assets/scaler.pkl")
joblib.dump(imputer, "../backend/assets/imputer_median.pkl")

print("Đã save model, scaler, imputer vào backend/assets/")

# Save reference stats cho radar chart (median của train set)
reference_stats = X_train.median().to_dict()
with open("../backend/assets/reference_stats.json", "w") as f:
    json.dump(reference_stats, f, indent=4)

print("Đã save reference_stats.json")

Đã save model, scaler, imputer vào backend/assets/
Đã save reference_stats.json
